In [3]:
!pip install sentence-transformers qdrant-client requests tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 10.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 45.2 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 38.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 48.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 61.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 42.8 MB/s  0:00:02m0:00:0100:01
  Attempting uninstall: regex
    Found existing installation: regex 2025.9.1
    Uninstalling regex-2025.9.1:
      Successfully uninstalled regex-2025.9.1
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: hf-xet0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/11 [numpy]
    Found existing installation: hf-xet 1.2.0━━━━━━━━━━━━━━━━━  3/11 [numpy]
    Uninstalling hf-xet-1.2.0:90m━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import gzip
import json
import requests
from datetime import datetime
import time
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct

In [3]:
date=datetime.now().strftime("%m_%d_%Y")
url=f"http://files.tmdb.org/p/exports/movie_ids_{date}.json.gz"

In [4]:
response=requests.get(url)

In [5]:
with open("movie_ids.json.gz", "wb") as f:
    f.write(response.content)

In [ ]:
movie_ids=[]
with gzip.open("movie_ids.json.gz", "rt") as f:
    for line in f:
        movie=json.loads(line)
        movie_ids.append(movie["id"])
print(f"Total movie IDs: {len(movie_ids)}")

In [ ]:
TMDB_API_KEY = os.environ.get("VITE_API_KEY")
def fetch_movie_details(movie_id):
    try:
        response=requests.get(f"https://api.themoviedb.org/3/movie/{movie_id}",params={"api_key": TMDB_API_KEY},timeout=5)
        if response.status_code==200:
            return response.json()
        return None
    except:
        return None
movies=[]
failed=[]
with ThreadPoolExecutor(max_workers=40) as executor:
    futures={executor.submit(fetch_movie_details, mid): mid for mid in movie_ids}
    for i,future in enumerate(tqdm(as_completed(futures),total=len(movie_ids))):
        result = future.result()
        if result:
            movies.append(result)
        else:
            failed.append(futures[future])
        if (i + 1) % 5000 == 0:
          with open("movies_progress.json", "w") as f:
            json.dump(movies, f)
          print(f"Saved {len(movies)} valid movies")
print(f"Successfully fetched: {len(movies)}")
print(f"Failed: {len(failed)}")

  0%|          | 5048/1196090 [01:15<5:33:30, 59.52it/s] 

Saved 4097 valid movies


  1%|          | 10058/1196090 [02:49<5:04:57, 64.82it/s] 

Saved 8101 valid movies


  1%|▏         | 15066/1196090 [04:35<5:55:18, 55.40it/s] 

...

Saved 881643 valid movies


100%|█████████▉| 1195000/1196090 [7:15:40<00:29, 37.25it/s]

Saved 885308 valid movies


100%|██████████| 1196090/1196090 [7:15:40<00:00, 45.76it/s]

Successfully fetched: 886069
Failed: 310021


In [8]:
#with open("movies_progress.json", "r") as f:
    #movies = json.load(f)
#print(f"Loaded {len(movies)} movies")

Loaded 885308 movies


In [ ]:
QDRANT_API_KEY=os.environ.get("QDRANT_API_KEY")
CLUSTER_ENDPOINT=os.environ.get("CLUSTER_ENDPOINT")
model = SentenceTransformer('all-MiniLM-L6-v2')
client = QdrantClient(url=CLUSTER_ENDPOINT, api_key=QDRANT_API_KEY)
def build_text(movie):
    genres = movie.get("genres", [])
    if isinstance(genres, list):
        genre_str=" ".join([g["name"] for g in genres if isinstance(g, dict)])
    else:
        genre_str=genres
    return f"{movie['title']} | {genre_str} | {movie['overview']}"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
movies=[m for m in movies if m.get("overview") and m.get("title")]

In [ ]:
def embed_and_upsert(movies,batch_size=100):
    total_upserted=0
    for i in tqdm(range(0,len(movies),batch_size)):
        batch=movies[i:i + batch_size]
        texts=[build_text(m) for m in batch]
        embeddings=model.encode(texts, show_progress_bar=False)
        points=[]
        for j,(movie, embedding) in enumerate(zip(batch,embeddings)):
            points.append(PointStruct(id=abs(hash(str(movie["id"]))) % (2**63),vector=embedding.tolist(),payload={"title": movie.get("title", ""),"overview": movie.get("overview", ""),"rating": float(movie.get("rating", movie.get("vote_average", 0))),"poster_path": movie.get("poster_path", ""),"genres": " ".join([g["name"] for g in movie.get("genres", [])]),"release_date": movie.get("release_date", "")}))
        client.upsert(collection_name="movies",points=points)
        total_upserted+=len(points)
        if (i // batch_size) % 50==0:
            print(f"Upserted {total_upserted} movies to Qdrant")
    return total_upserted
total=embed_and_upsert(movies)
print(f"Done loading {total} movies in Qdrant")

  0%|          | 1/8854 [00:03<8:15:27,  3.36s/it]

Upserted 100 movies to Qdrant


  1%|          | 51/8854 [00:52<2:02:31,  1.20it/s]

Upserted 5100 movies to Qdrant


  1%|          | 101/8854 [01:26<1:35:50,  1.52it/s]

...

Upserted 880100 movies to Qdrant


100%|█████████▉| 8851/8854 [1:47:37<00:02,  1.35it/s]

Upserted 885100 movies to Qdrant


100%|██████████| 8854/8854 [1:47:39<00:00,  1.37it/s]

Done loading 885308 movies in Qdrant


In [12]:
collection_info=client.get_collection("movies")
print(collection_info)

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=880900 points_count=885308 segments_count=7 config=CollectionConfig(params=CollectionParams(vectors={'': VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=HnswConfigDiff(m=24, ef_construct=256, full_scan_threshold=None, max_indexing_threads=None, on_disk=None, payload_m=24, inline_storage=None), quantization_config=None, on_disk=False, datatype=<Datatype.FLOAT32: 'float32'>, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_numb